In [11]:
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from genkit._noise import sample_scaled_isotropic_alpha_stable
from genkit.diffusion import DDPMV, DLPMEps
from genkit.metrics import mmd_rbf
from genkit.nn import MLPModel
from genkit.training import train


DEVICE = torch.device("cpu")
DTYPE = torch.float32
SEED = 0
N_TRIALS = 3

ALPHA = 1.7
DIM = 1
N_TRAIN = 100
N_TEST = N_MMD = 2_000
N_MIXTURE_SCALES = 4_095
N_MOM_BLOCKS = 9  # Median-of-means blocks; must be an odd divisor of N_MIXTURE_SCALES.
SAMPLE_CHUNK_SIZE = 1_000
ORACLE_BATCH_SIZE = 500
ORACLE_TRAIN_CHUNK_SIZE = 100
ORACLE_SCALE_CHUNK_SIZE = 455

N_REVERSE_STEPS = 128
DDPM_N_STEPS = N_REVERSE_STEPS
DLPM_N_STEPS = N_REVERSE_STEPS + 1  # DLPM's native sampler runs n_steps - 1 reverse updates.
DDPM_SIGMA_MAX = 5.0

N_EPOCHS = 128
BATCH_SIZE = 128
LR = 5e-4
DEPTH = 3
WIDTH = 64

TRAIN_KWARGS = dict(batch_size=BATCH_SIZE, n_epochs=N_EPOCHS, lr=LR, device=DEVICE, use_adamw=False, lr_schedule="constant", freq_logging=0)


In [12]:
def sample_alpha_stable(n_samples):
    return sample_scaled_isotropic_alpha_stable(n_samples, dim=DIM, alpha=ALPHA, device=DEVICE, dtype=DTYPE)


def make_net(output_dim=DIM):
    return MLPModel(input_dim=DIM, output_dim=output_dim, width=WIDTH, depth=DEPTH).to(device=DEVICE, dtype=DTYPE)


def set_seed(seed):
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))


@torch.no_grad()
def evaluate_samples(name, trial, x_ref, x_gen):
    x_ref = x_ref.detach().cpu().to(torch.float64)
    x_gen = x_gen.detach().cpu().to(torch.float64)

    return {"trial": trial, "model": name, "mmd_rbf": mmd_rbf(x_ref[:N_MMD], x_gen[:N_MMD])}


def evaluate_sampler(name, sample_fn, trial, x_test):
    chunks = []

    for start in range(0, len(x_test), SAMPLE_CHUNK_SIZE):
        size = min(SAMPLE_CHUNK_SIZE, len(x_test) - start)
        chunks.append(sample_fn(size).detach().cpu())

    return evaluate_samples(name, trial, x_test, torch.cat(chunks))


class _EmpiricalOracle:
    def _set_x_train(self, x_train):
        x_train = x_train.to(device=self._device, dtype=self._fdtype)

        if x_train.ndim < 2 or tuple(x_train.shape[1:]) != self._sample_shape:
            raise ValueError(f"x_train must have shape (N, *{self._sample_shape}), got {tuple(x_train.shape)}")

        self._x_train = x_train.flatten(1)
        self._x_train_sq = self._x_train.square().sum(dim=1)

    def _dist2(self, x, scale, x_train=None, x_train_sq=None):
        x = x.flatten(1)
        x_train = self._x_train if x_train is None else x_train
        x_train_sq = self._x_train_sq if x_train_sq is None else x_train_sq

        return (
            x.square().sum(dim=1, keepdim=True)
            + scale.square() * x_train_sq
            - 2.0 * scale * (x @ x_train.T)
        ).clamp_min(0.0)

    def _mean_x0(self, weights, like):
        return (weights @ self._x_train).reshape_as(like)


class CF_DDPMV(_EmpiricalOracle, DDPMV):
    """Exact empirical DDPM oracle using the existing DDPM/DDIM sampler."""

    def __init__(self, x_train, **kwargs):
        dim = x_train.shape[1] if x_train.ndim == 2 else tuple(x_train.shape[1:])
        super().__init__(net=torch.nn.Identity(), dim=dim, **kwargs)
        self._set_x_train(x_train)

    def _score(self, x, t_idx):
        alpha_bar = self._alpha_bar[t_idx]
        scale = alpha_bar.sqrt()
        variance = (1.0 - alpha_bar).clamp_min(self._eps) * self._sigma_max**2

        weights = torch.softmax(-self._dist2(x, scale) / (2.0 * variance), dim=1)
        x0_hat = self._mean_x0(weights, x)

        return (scale * x0_hat - x) / variance

    def _get_eps_hat(self, x, t_norm, t_idx):
        sigma = (1.0 - self._alpha_bar[t_idx]).clamp_min(self._eps).sqrt()
        return -sigma * self._sigma_max**2 * self._score(x, t_idx)


class CF_DLPMEps(_EmpiricalOracle, DLPMEps):
    """
    Empirical DLPM epsilon oracle based on a finite Gaussian scale mixture.

    n_mom_blocks=1 uses the ordinary Monte Carlo mean.
    n_mom_blocks>1 uses a median-of-means likelihood estimator.
    """

    def __init__(self, x_train, n_mixture_scales=4095, n_mom_blocks=9, **kwargs):
        dim = x_train.shape[1] if x_train.ndim == 2 else tuple(x_train.shape[1:])
        super().__init__(net=torch.nn.Identity(), dim=dim, **kwargs)
        self._set_x_train(x_train)

        n_mixture_scales = int(n_mixture_scales)
        n_mom_blocks = int(n_mom_blocks)

        if n_mixture_scales < 1:
            raise ValueError("n_mixture_scales must be positive.")
        if n_mom_blocks < 1 or n_mom_blocks % 2 == 0:
            raise ValueError("n_mom_blocks must be a positive odd integer.")
        if n_mixture_scales % n_mom_blocks:
            raise ValueError("n_mixture_scales must be divisible by n_mom_blocks.")

        block_size = n_mixture_scales // n_mom_blocks
        mix_A = self._draw_A(n_mixture_scales).reshape(n_mom_blocks, block_size).clamp_min(self._eps)

        self._mix_A = mix_A
        self._mix_log_A = mix_A.log()

    def _block_log_mean(self, dist2, sigma2, A, log_A):
        """Log Monte Carlo mean of the conditional Gaussian likelihood over one block."""

        log_sum = None

        for A_chunk, log_A_chunk in zip(
            A.split(ORACLE_SCALE_CHUNK_SIZE), log_A.split(ORACLE_SCALE_CHUNK_SIZE)
        ):
            A_chunk = A_chunk.view(1, 1, -1)
            log_A_chunk = log_A_chunk.view(1, 1, -1)

            logits = -0.5 * self._dim * log_A_chunk - dist2.unsqueeze(-1) / (2.0 * sigma2 * A_chunk)
            chunk_log_sum = logits.logsumexp(dim=-1)
            log_sum = chunk_log_sum if log_sum is None else torch.logaddexp(log_sum, chunk_log_sum)

        return log_sum - math.log(A.numel())

    def _log_likelihood(self, dist2, sigma2):
        block_log_means = [
            self._block_log_mean(dist2, sigma2, A, log_A)
            for A, log_A in zip(self._mix_A, self._mix_log_A)
        ]

        return torch.stack(block_log_means, dim=-1).median(dim=-1).values

    def _posterior_mean_x0(self, x, gamma, sigma):
        sigma2 = sigma.square().clamp_min(self._eps)
        posterior_means = []

        for x_block in x.split(ORACLE_BATCH_SIZE):
            log_weight_chunks = []

            for start in range(0, len(self._x_train), ORACLE_TRAIN_CHUNK_SIZE):
                stop = min(start + ORACLE_TRAIN_CHUNK_SIZE, len(self._x_train))
                x_train = self._x_train[start:stop]
                x_train_sq = self._x_train_sq[start:stop]

                dist2 = self._dist2(x_block, gamma, x_train, x_train_sq)
                log_weight_chunks.append(self._log_likelihood(dist2, sigma2))

            weights = torch.softmax(torch.cat(log_weight_chunks, dim=1), dim=1)
            posterior_means.append(self._mean_x0(weights, x_block))

        return torch.cat(posterior_means)

    def _get_eps_hat(self, x, t):
        gamma = self._gamma_1_t[t]
        sigma = self._sigma_1_t[t].clamp_min(self._eps)
        x0_hat = self._posterior_mean_x0(x, gamma, sigma)

        return (x - gamma * x0_hat) / sigma

    @torch.no_grad()
    def _sample(self, n_samples, return_trajectory=True):
        self._net.eval()

        A_path = torch.stack([self._draw_A(n_samples).reshape(-1) for _ in range(self._n_steps)])
        Sigma_1_t = self._Sigma_1_t(A_path)

        x = self._sigma_1_t[self._n_steps - 1] * self._sample_source_default(n_samples)
        trajectory = [x] if return_trajectory else None

        for t in range(self._n_steps - 1, 0, -1):
            Sigma_hat, gamma_t, Gamma_t = self._g_Sigma_hat_Gamma(Sigma_1_t, t)
            eps_hat = self._get_eps_hat(x, t)

            Gamma_t = self._expand_batch_scalar(Gamma_t, x)
            x = (x - Gamma_t * self._sigma_1_t[t] * eps_hat) / gamma_t

            if t > 1:
                noise = torch.randn_like(x)
                x += self._expand_batch_scalar(Sigma_hat.sqrt(), noise) * noise

            if return_trajectory:
                trajectory.append(x)

        return x, trajectory

In [ ]:
set_seed(SEED)
x_test = sample_alpha_stable(N_TEST)
x_train = sample_alpha_stable(N_TRAIN)

rows = []
for trial in range(1, N_TRIALS + 1):
    print(f"Trial {trial} / {N_TRIALS}:", end="")
    trial_seed = SEED + 10_000 * trial

    start = time.time()
    set_seed(trial_seed + 1)
    ddpm = DDPMV(net=make_net(), dim=DIM, n_steps=DDPM_N_STEPS, sigma_max=DDPM_SIGMA_MAX, sampler="ddpm", device=DEVICE, fdtype=DTYPE)
    ddpm, _ = train(ddpm, x_train, **TRAIN_KWARGS)
    set_seed(trial_seed + 2)
    rows.append(evaluate_sampler("DDPM", ddpm.sample, trial, x_test))
    print(f" 'DDPM' done in {time.time() - start:.1f}s", end="")

    start = time.time()
    set_seed(trial_seed + 3)
    cf_ddpm = CF_DDPMV(x_train=x_train, n_steps=DDPM_N_STEPS, sigma_max=DDPM_SIGMA_MAX, device=DEVICE, fdtype=DTYPE)
    rows.append(evaluate_sampler("CF-DDPM", cf_ddpm.sample, trial, x_test))
    print(f" 'CF-DDPM' done in {time.time() - start:.1f}s", end="")

    start = time.time()
    set_seed(trial_seed + 4)
    dlpm = DLPMEps(net=make_net(), dim=DIM, n_steps=DLPM_N_STEPS, alpha=ALPHA, device=DEVICE, fdtype=DTYPE)
    dlpm, _ = train(dlpm, x_train, **TRAIN_KWARGS)
    set_seed(trial_seed + 5)
    rows.append(evaluate_sampler("DLPM", dlpm.sample, trial, x_test))
    print(f" 'DLPM' done in {time.time() - start:.1f}s", end="")

    start = time.time()
    set_seed(trial_seed + 6)
    cf_dlpm = CF_DLPMEps(x_train=x_train, n_steps=DLPM_N_STEPS, alpha=ALPHA, n_mixture_scales=N_MIXTURE_SCALES, n_mom_blocks=N_MOM_BLOCKS, device=DEVICE, fdtype=DTYPE)
    set_seed(trial_seed + 7)
    rows.append(evaluate_sampler("CF-DLPM", cf_dlpm.sample, trial, x_test))
    print(f" 'CF-DLPM' done in {time.time() - start:.1f}s", end="")

    start = time.time()
    set_seed(trial_seed + 8)
    rows.append(evaluate_samples("test-vs-test", trial, x_test, sample_alpha_stable(len(x_test))))
    print(f", 'test-vs-test' done in {time.time() - start:.1f}s", end="\n")



Trial 1 / 3: 'DDPM' done in 1.2s 'CF-DDPM' done in 0.4s 'DLPM' done in 1.4s

In [ ]:
results = pd.DataFrame(rows)
results["model"] = results["model"].replace({"test vs true sample": "test-vs-test"})
mmd_table = results.groupby("model", sort=False)["mmd_rbf"].agg(["mean", "std"]).fillna(0.0)
mmd_table = mmd_table.rename(columns={"mean": "mmd_rbf_mean", "std": "mmd_rbf_std"})

baseline_name = "test-vs-test"
baseline_mean = float(mmd_table.loc[baseline_name, "mmd_rbf_mean"]) if baseline_name in mmd_table.index else None
bar_table = mmd_table.drop(index=baseline_name, errors="ignore")

labels = bar_table.index.to_list()
means = bar_table["mmd_rbf_mean"].to_numpy(dtype=float)
errors = bar_table["mmd_rbf_std"].to_numpy(dtype=float)
plot_means = np.clip(means, 1e-12, None)
plot_errors = np.vstack([np.minimum(errors, 0.9 * plot_means), errors])

best_idx = int(np.argmin(plot_means))
best_label = labels[best_idx]
best_mean = float(plot_means[best_idx])
x = np.arange(len(labels))

colors = []
for label in labels:
    if "DDPM" in label:
        colors.append("#2f6fbb")
    elif "DLPM" in label:
        colors.append("#d9792b")
    else:
        colors.append("#5b5b5b")


In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 2.8))

bars = ax.bar(x, plot_means, yerr=plot_errors, capsize=4, width=0.45, color=colors, edgecolor="#202020", linewidth=1.4, zorder=3)
for bar, label in zip(bars, labels):
    if label.startswith("CF-"):
        bar.set_hatch("///")
        bar.set_alpha(0.85)

ax.axhline(baseline_mean, color="#5b5b5b", linestyle="--", linewidth=1.8, zorder=4)
ax.axhline(best_mean, color="#202020", linewidth=1.5, alpha=0.8, zorder=4)

ax.set_yscale("log")
ax.set_title("MMD-RBF")
ax.set_xticks(x, labels, rotation=20, ha="right")
ax.grid(axis="y", which="both", alpha=0.25, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
plt.show()
